In [1]:
# ============================================================
# PS S6E5 - exp_20260516_033_blend_logreg_topk_feature_variants_min
#
# D: blend-code small improvements
# - Avg / HC / NM / Signed SLSQP on raw OOF predictions
# - Prune weak models
# - Ridge / LogReg on current raw+rank+logit meta features
# - Additional LogReg feature variants: logit_only, raw_logit, raw_only
# - Additional LogReg topK equal blends from C-search candidates
# - Additional Ridge feature variants: logit_only, raw_logit
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260516_033_blend_logreg_topk_feature_variants_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)
RUN_D_EXTRA = False

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================
#
# D experiment stack:
# - Drop 030/031 because shared011 CatBoost meta-stacking did not improve Public LB.
# - Drop old RealMLP variants 007/022/023/027.
# - Drop 018 by default because recent optimized blends effectively zeroed it out after 029/032.
# - Keep 014/015/016/017/021/026/028/029/032.
#
# If you want a conservative comparison, re-add 018 manually and rerun.
# ============================================================

MODEL_SPECS = [
    # {
    #     "key": "014",
    #     "name": "cat_shared004_no_tyreageratio_str_min",
    #     "family": "CatBoost",
    #     "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
    #     "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
    #     "public_lb": 0.95233,
    # },
    # {
    #     "key": "015",
    #     "name": "cat_shared004_no_race_year_groupstats_min",
    #     "family": "CatBoost",
    #     "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
    #     "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
    #     "public_lb": 0.95244,
    # },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    # {
    #     "key": "017",
    #     "name": "xgb_shared005_no_pitstop_pairte_min",
    #     "family": "XGBoost",
    #     "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
    #     "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
    #     "public_lb": 0.95164,
    # },
    {
        "key": "021",
        "name": "tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
        "family": "TabM",
        "oof": "oof_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "public_lb": 0.95171,
    },
    # {
    #     "key": "026",
    #     "name": "lgb_weather_year_stint_flags_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
    #     "pred": "pred_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
    #     "public_lb": 0.95096,
    # },
    {
        "key": "028",
        "name": "parent_child_mlp_shared010_min",
        "family": "Parent-Child MLP",
        "oof": "oof_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "pred": "pred_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "public_lb": 0.95260,
    },
    # {
    #     "key": "029",
    #     "name": "realmlp_shared001v2_min",
    #     "family": "RealMLP",
    #     "oof": "oof_exp_20260514_029_realmlp_shared001v2_min.npy",
    #     "pred": "pred_exp_20260514_029_realmlp_shared001v2_min.npy",
    #     "public_lb": 0.95397,
    # },
    # {
    #     "key": "031",
    #     "name": "catboost_meta_stacking_shared011_tawara_gpu_min",
    #     "family": "CatBoost",
    #     "oof": "oof_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
    #     "pred": "pred_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
    #     "public_lb": 0.95220,
    # },
    {
        "key": "032",
        "name": "custom_realmlp_cv095352_min",
        "family": "CustomTorchRealMLP",
        "oof": "oof_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "pred": "pred_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "public_lb": 0.95309,
    },
    # {
    #     "key": "034",
    #     "name": "lgb_laptime_delta_nonlinear_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260517_034_lgb_laptime_delta_nonlinear_min.npy",
    #     "pred": "pred_exp_20260517_034_lgb_laptime_delta_nonlinear_min.npy",
    #     "public_lb": 0.95089,
    # },
    # {
    #     "key": "035",
    #     "name": "xgb_laptime_delta_nonlinear_min",
    #     "family": "XGBoost",
    #     "oof": "oof_exp_20260518_035_xgb_laptime_delta_nonlinear_min.npy",
    #     "pred": "pred_exp_20260518_035_xgb_laptime_delta_nonlinear_min.npy",
    #     "public_lb": 0.95162,
    # },
    {
        "key": "036",
        "name": "realmlp_shared001v2_no_race_year_te_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260518_036_realmlp_shared001v2_no_race_year_te_min.npy",
        "pred": "pred_exp_20260518_036_realmlp_shared001v2_no_race_year_te_min.npy",
        "public_lb": 0.95393,
    },
    # {
    #     "key": "037",
    #     "name": "realmlp_shared001v2_pitstop_light_min",
    #     "family": "RealMLP",
    #     "oof": "oof_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy",
    #     "pred": "pred_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy",
    #     "public_lb": 0.95394,
    # },
    # {
    #     "key": "038",
    #     "name": "realmlp_shared001v2_seed_reg_variant_min",
    #     "family": "RealMLP",
    #     "oof":   "oof_exp_20260518_038_realmlp_shared001v2_seed_reg_variant_min.npy",
    #     "pred": "pred_exp_20260518_038_realmlp_shared001v2_seed_reg_variant_min.npy",
    #     "public_lb": 0.95384,
    # },
    {
        "key": "039",
        "name": "realmlp_shared001v2_log_features_seedreg_min",
        "family": "RealMLP",
        "oof":   "oof_exp_20260519_039_realmlp_shared001v2_log_features_seedreg_min.npy",
        "pred": "pred_exp_20260519_039_realmlp_shared001v2_log_features_seedreg_min.npy",
        "public_lb": 0.95375,
    },
    {
        "key": "040",
        "name": "lgb_shared001v2_features_min",
        "family": "LightGBM",
        "oof":   "oof_exp_20260519_040_lgb_shared001v2_features_min.npy",
        "pred": "pred_exp_20260519_040_lgb_shared001v2_features_min.npy",
        "public_lb": 0.95195,
    },
    # {
    #     "key": "041",
    #     "name": "xgb_shared001v2_features_min",
    #     "family": "XGBoost",
    #     "oof":   "oof_exp_20260520_041_xgb_shared001v2_features_min.npy",
    #     "pred": "pred_exp_20260520_041_xgb_shared001v2_features_min.npy",
    #     "public_lb": 0.95173,
    # },
    {
        "key": "042",
        "name": "cat_shared001v2_features_min",
        "family": "CatBoost",
        "oof":   "oof_exp_20260520_042_cat_shared001v2_features_min.npy",
        "pred": "pred_exp_20260520_042_cat_shared001v2_features_min.npy",
        "public_lb": 0.95283,
    },
    {
        "key": "044",
        "name": "realmlp_shared001v2_seed_reg_variant2_min",
        "family": "RealMLP",
        "oof":   "oof_exp_20260520_044_realmlp_shared001v2_seed_reg_variant2_min.npy",
        "pred": "pred_exp_20260520_044_realmlp_shared001v2_seed_reg_variant2_min.npy",
        "public_lb": 0.95381,
    },
    {
        "key": "046",
        "name": "cat_shared001v2_features_optuna_refit_min",
        "family": "CatBoost",
        "oof":   "oof_exp_20260520_046_cat_shared001v2_features_optuna_refit_min.npy",
        "pred": "pred_exp_20260520_046_cat_shared001v2_features_optuna_refit_min.npy",
        "public_lb": 0.95283,
    },
    {
        "key": "048",
        "name": "lgb_shared001v2_features_optuna_refit_min",
        "family": "LightGBM",
        "oof":   "oof_exp_20260521_048_lgb_shared001v2_features_optuna_refit_min.npy",
        "pred": "pred_exp_20260521_048_lgb_shared001v2_features_optuna_refit_min.npy",
        "public_lb": 0.95191,
    },
    {
        "key": "050",
        "name": "xgb_shared001v2_features_optuna_refit_min",
        "family": "XGBoost",
        "oof":   "oof_exp_20260524_050_xgb_shared001v2_features_optuna_refit_min.npy",
        "pred": "pred_exp_20260524_050_xgb_shared001v2_features_optuna_refit_min.npy",
        "public_lb": 0.95223,
    },
    {
        "key": "051",
        "name": "realmlp_watchlist_updated_shared001v2_min",
        "family": "Realmlp",
        "oof":   "oof_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.npy",
        "pred": "pred_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.npy",
        "public_lb": 0.95407,
    },
    {
        "key": "054",
        "name": "realmlp_watchlist_updated_optuna_refit_min",
        "family": "Realmlp",
        "oof":   "oof_exp_20260525_054_realmlp_watchlist_updated_optuna_refit_min.npy",
        "pred": "pred_exp_20260525_054_realmlp_watchlist_updated_optuna_refit_min.npy",
        "public_lb": 0.95389,
    },
    {
        "key": "055",
        "name": "realmlp_watchlist_updated_optuna_refit_epoch6_min",
        "family": "Realmlp",
        "oof":   "oof_exp_20260525_055_realmlp_watchlist_updated_optuna_refit_epoch6_min.npy",
        "pred": "pred_exp_20260525_055_realmlp_watchlist_updated_optuna_refit_epoch6_min.npy",
        "public_lb": 0.95390,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("model_keys:", model_keys)
print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)


model_keys: ['016', '021', '028', '032', '036', '039', '040', '042', '044', '046', '048', '050', '051', '054', '055']
X_raw: (439140, 15)
T_raw: (188165, 15)


In [7]:
# model_specs sanity check
for spec in MODEL_SPECS:
    key = spec["key"]
    oof = spec["oof"]
    pred = spec["pred"]

    if key not in oof:
        print(f"[WARN] key {key} not found in oof path: {oof}")
    if key not in pred:
        print(f"[WARN] key {key} not found in pred path: {pred}")

In [8]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
5,039,realmlp_shared001v2_log_features_seedreg_min,RealMLP,0.954146,0.95375,4.635888e-06,0.999097,0.000006,0.999130
12,051,realmlp_watchlist_updated_shared001v2_min,Realmlp,0.954093,0.95407,4.593853e-06,0.999267,0.000006,0.999016
4,036,realmlp_shared001v2_no_race_year_te_min,RealMLP,0.954075,0.95393,3.469280e-06,0.999499,0.000007,0.999095
13,054,realmlp_watchlist_updated_optuna_refit_min,Realmlp,0.954067,0.95389,7.995004e-06,0.999310,0.000010,0.998928
8,044,realmlp_shared001v2_seed_reg_variant2_min,RealMLP,0.953996,0.95381,5.493830e-06,0.999279,0.000007,0.999363
14,055,realmlp_watchlist_updated_optuna_refit_epoch6_min,Realmlp,0.953908,0.95390,4.421508e-06,0.999543,0.000019,0.997728
3,032,custom_realmlp_cv095352_min,CustomTorchRealMLP,0.953506,0.95309,3.416236e-04,0.998800,0.000474,0.998323
9,046,cat_shared001v2_features_optuna_refit_min,CatBoost,0.953361,0.95283,3.509111e-05,0.997057,0.000117,0.996167
7,042,cat_shared001v2_features_min,CatBoost,0.953241,0.95283,2.908368e-05,0.997476,0.000146,0.996312
10,048,lgb_shared001v2_features_optuna_refit_min,LightGBM,0.952455,0.95191,2.183121e-05,0.998960,0.000037,0.999256


best single: 039 0.9541459595754959


In [9]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,016,021,028,032,036,039,040,042,044,046,048,050,051,054,055
016,1.000000,0.978284,0.972366,0.979115,0.979401,0.977873,0.965045,0.963893,0.978122,0.964340,0.962436,0.997457,0.979795,0.979787,0.981250
021,0.978284,1.000000,0.974491,0.982081,0.979649,0.979618,0.969504,0.968171,0.979760,0.968594,0.967722,0.978314,0.980211,0.980663,0.982017
028,0.972366,0.974491,1.000000,0.979124,0.980752,0.980895,0.958662,0.957683,0.980841,0.958077,0.956407,0.973403,0.980530,0.981187,0.981581
032,0.979115,0.982081,0.979124,1.000000,0.990511,0.990135,0.962340,0.961143,0.990754,0.961548,0.959585,0.980146,0.990446,0.990823,0.990435
036,0.979401,0.979649,0.980752,0.990511,1.000000,0.995111,0.961328,0.961383,0.995553,0.961756,0.958420,0.981540,0.997585,0.997738,0.996291
039,0.977873,0.979618,0.980895,0.990135,0.995111,1.000000,0.960904,0.959530,0.995253,0.959875,0.958046,0.979970,0.994493,0.994894,0.993543
040,0.965045,0.969504,0.958662,0.962340,0.961328,0.960904,1.000000,0.986801,0.960803,0.987379,0.998600,0.964624,0.961815,0.961878,0.964939
042,0.963893,0.968171,0.957683,0.961143,0.961383,0.959530,0.986801,1.000000,0.959854,0.997946,0.987127,0.963258,0.961880,0.961803,0.965120
044,0.978122,0.979760,0.980841,0.990754,0.995553,0.995253,0.960803,0.959854,1.000000,0.960236,0.957906,0.980061,0.994954,0.995267,0.993992
046,0.964340,0.968594,0.958077,0.961548,0.961756,0.959875,0.987379,0.997946,0.960236,1.000000,0.987714,0.963628,0.962245,0.962168,0.965633


Spearman correlation


,016,021,028,032,036,039,040,042,044,046,048,050,051,054,055
016,1.000000,0.965983,0.967417,0.966370,0.971731,0.970770,0.971928,0.976492,0.969946,0.977357,0.973263,0.995796,0.972220,0.972823,0.972946
021,0.965983,1.000000,0.967054,0.966383,0.968109,0.970407,0.963786,0.969489,0.968850,0.969970,0.966249,0.967741,0.968692,0.970320,0.968386
028,0.967417,0.967054,1.000000,0.967367,0.972625,0.974987,0.962161,0.969298,0.973479,0.969929,0.964642,0.970794,0.972489,0.974490,0.973204
032,0.966370,0.966383,0.967367,1.000000,0.981078,0.979910,0.966727,0.972668,0.981575,0.973201,0.969231,0.968434,0.980053,0.981302,0.978191
036,0.971731,0.968109,0.972625,0.981078,1.000000,0.990057,0.970607,0.977486,0.991993,0.978194,0.972530,0.973792,0.995426,0.995795,0.991543
039,0.970770,0.970407,0.974987,0.979910,0.990057,1.000000,0.971570,0.975993,0.990522,0.976586,0.973907,0.973094,0.989024,0.989945,0.986128
040,0.971928,0.963786,0.962161,0.966727,0.970607,0.971570,1.000000,0.974004,0.970547,0.974740,0.997430,0.971220,0.970525,0.970295,0.968063
042,0.976492,0.969489,0.969298,0.972668,0.977486,0.975993,0.974004,1.000000,0.975879,0.996441,0.974720,0.977304,0.977012,0.977304,0.974725
044,0.969946,0.968850,0.973479,0.981575,0.991993,0.990522,0.970547,0.975879,1.000000,0.976575,0.972563,0.972232,0.990937,0.991771,0.987914
046,0.977357,0.969970,0.969929,0.973201,0.978194,0.976586,0.974740,0.996441,0.976575,1.000000,0.975308,0.977949,0.977720,0.977992,0.975956


In [10]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T, keys):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in keys] +
        [f"rank_{k}" for k in keys] +
        [f"logit_{k}" for k in keys]
    )

    return X_meta, T_meta, feature_names


X_meta_full, T_meta_full, meta_feature_names_full = build_meta_features(
    X_raw,
    T_raw,
    model_keys,
)

print("X_meta_full:", X_meta_full.shape)
print("T_meta_full:", T_meta_full.shape)
print("meta meta_feature_names_full:", meta_feature_names_full)

def build_meta_features_variant(X, T, keys, variant="raw_rank_logit"):
    """Build meta features with explicit feature-set variant.

    Variants:
      - raw_rank_logit: current baseline
      - raw_logit
      - logit_only
      - raw_only
      - rank_only
      - raw_rank
      - rank_logit
    """
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    parts_X = []
    parts_T = []
    names = []

    def add_part(prefix, X_part, T_part):
        parts_X.append(X_part)
        parts_T.append(T_part)
        names.extend([f"{prefix}_{k}" for k in keys])

    if variant == "raw_rank_logit":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_logit":
        add_part("raw", X, T)
        add_part("logit", X_logit, T_logit)
    elif variant == "logit_only":
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_only":
        add_part("raw", X, T)
    elif variant == "rank_only":
        add_part("rank", X_rank, T_rank)
    elif variant == "raw_rank":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
    elif variant == "rank_logit":
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    else:
        raise ValueError(f"Unknown meta feature variant: {variant}")

    return np.column_stack(parts_X), np.column_stack(parts_T), names


X_meta_full: (439140, 45)
T_meta_full: (188165, 45)
meta meta_feature_names_full: ['raw_016', 'raw_021', 'raw_028', 'raw_032', 'raw_036', 'raw_039', 'raw_040', 'raw_042', 'raw_044', 'raw_046', 'raw_048', 'raw_050', 'raw_051', 'raw_054', 'raw_055', 'rank_016', 'rank_021', 'rank_028', 'rank_032', 'rank_036', 'rank_039', 'rank_040', 'rank_042', 'rank_044', 'rank_046', 'rank_048', 'rank_050', 'rank_051', 'rank_054', 'rank_055', 'logit_016', 'logit_021', 'logit_028', 'logit_032', 'logit_036', 'logit_039', 'logit_040', 'logit_042', 'logit_044', 'logit_046', 'logit_048', 'logit_050', 'logit_051', 'logit_054', 'logit_055']


In [11]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
    weight_keys=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "weight_keys": None if weights is None else list(weight_keys or model_keys),
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        f"delta_vs_{best_single_key}": float(score - auc(y, oofs[best_single_key])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [12]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_014_015_016_018_020_021_022_023": ["007", "014", "015", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_016_018_020_021_022_023": ["007", "014", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_015_018_020_021_022_023": ["007", "014", "015", "018", "020", "021", "022", "023"],
    "avg" : model_keys
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg: 0.954945178


In [13]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.955025964
HC weights:
016 0.027615596627950597
021 0.013061894793253957
028 0.035236504857546466
032 0.10303847204213913
036 0.027789283614942036
039 0.23503859220031903
040 0.02773197916690618
042 0.07658454394861576
044 0.09978474050089159
046 0.08657868989212711
048 0.038628690417288045
050 0.0899123149487306
051 0.1292235593960666
054 0.0
055 0.009775137593222809


In [14]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.955026
         Iterations: 276
         Function evaluations: 697
nm_softmax_raw: 0.955025990
NM weights:
016 0.027791719641403145
021 0.01309051747015596
028 0.03530104365591787
032 0.10275839260307766
036 0.027964912363758473
039 0.23518428256865492
040 0.027815959181871318
042 0.07644337491829947
044 0.09971012555149333
046 0.08672468769239447
048 0.03848996092649679
050 0.0897473989917918
051 0.12929401711885288
054 9.045094960565579e-09
055 0.009683598270737082


In [15]:
# ============================================================
# Signed SLSQP weights
# Allow small negative weights, with sum(w)=1
# ============================================================

def weighted_average_signed(X, w):
    return np.asarray(X, dtype=float) @ np.asarray(w, dtype=float)


def optimize_signed_slsqp(
    X,
    y,
    start_w=None,
    neg_bound=-0.10,
    pos_bound=0.60,
    maxiter=1000,
):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n
    else:
        start_w = np.asarray(start_w, dtype=float)

    # Start must satisfy sum=1
    start_w = start_w / start_w.sum()

    bounds = [(neg_bound, pos_bound) for _ in range(n)]

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ]

    def objective(w):
        p = weighted_average_signed(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        start_w,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": 1e-12,
            "disp": True,
        },
    )

    w = res.x
    score = auc(y, weighted_average_signed(X, w))

    return w, score, res

In [16]:
print("\n" + "=" * 80)
print("Signed SLSQP")
print("=" * 80)

signed_w, signed_score, signed_res = optimize_signed_slsqp(
    X_raw,
    y,
    start_w=nm_w,
    neg_bound=-0.05,
    pos_bound=0.50,
    maxiter=1000,
)

add_candidate(
    name="slsqp_signed_raw_neg005",
    method="slsqp_signed",
    oof_pred=weighted_average_signed(X_raw, signed_w),
    test_pred=weighted_average_signed(T_raw, signed_w),
    weights=signed_w,
    params={
        "constraint": "sum1_signed",
        "neg_bound": -0.05,
        "pos_bound": 0.50,
        "success": bool(signed_res.success),
        "message": str(signed_res.message),
        "nit": int(getattr(signed_res, "nit", -1)),
        "fun": float(signed_res.fun),
    },
    notes="signed SLSQP on raw OOF predictions; high overfit risk; attack-only",
)

print("Signed SLSQP weights:")
for k, w in zip(model_keys, signed_w):
    print(k, w)

print("Signed score:", signed_score)
print("sum weights:", signed_w.sum())
print("min weight:", signed_w.min())
print("max weight:", signed_w.max())


Signed SLSQP
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.9550259476746308
            Iterations: 17
            Function evaluations: 441
            Gradient evaluations: 17
slsqp_signed_raw_neg005: 0.955025948
Signed SLSQP weights:
016 0.02787003141357311
021 0.0131080222586164
028 0.03542787215993146
032 0.10263113581852436
036 0.027980070296840694
039 0.23534626816524495
040 0.02790626041446989
042 0.07646204453455453
044 0.09963713815926031
046 0.08642394253143022
048 0.03834933068554784
050 0.090006948414186
051 0.12931104742915345
054 -3.260771347786767e-05
055 0.009572495432144598
Signed score: 0.9550259476746308
sum weights: 0.9999999999999999
min weight: -3.260771347786767e-05
max weight: 0.23534626816524495


In [17]:
# ============================================================
# Prune models based on HC / NM / SLSQP weights
# ============================================================

PRUNE_HARD_THRESHOLD = 0.005
PRUNE_SOFT_THRESHOLD = 0.020
KEEP_PROTECTED_KEYS = {
                    # CatBoost branch　"014", "015",
    "050",          # XGB branch; 017 is historically useful for LogReg "016", "017",
    "021",          # TabM branch
    "034",          # LGB branch
    "028",          # structurally different NN branch
    "029",          # current main RealMLP representative
    "032",          # custom Torch RealMLP auxiliary branch
}

def build_pruned_model_keys(model_keys, hc_w, nm_w, signed_w):
    rows = []
    for k, wh, wn, ws in zip(model_keys, hc_w, nm_w, signed_w):
        max_w = max(float(wh), float(wn), float(ws))
        min_w = min(float(wh), float(wn), float(ws))

        if max_w < PRUNE_HARD_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_hard"
        elif max_w < PRUNE_SOFT_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_soft"
        else:
            decision = "keep"

        rows.append({
            "key": k,
            "hc_weight": float(wh),
            "nm_weight": float(wn),
            "slsqp_weight": float(ws),
            "max_weight": max_w,
            "min_weight": min_w,
            "protected": k in KEEP_PROTECTED_KEYS,
            "decision": decision,
        })

    prune_df = pd.DataFrame(rows)
    keep_keys = prune_df.loc[prune_df["decision"] == "keep", "key"].tolist()
    drop_keys = prune_df.loc[prune_df["decision"] != "keep", "key"].tolist()

    return keep_keys, drop_keys, prune_df


keep_keys, drop_keys, prune_df = build_pruned_model_keys(
    model_keys=model_keys,
    hc_w=hc_w,
    nm_w=nm_w,
    signed_w=signed_w,
)

print("keep_keys:", keep_keys)
print("drop_keys:", drop_keys)
display(prune_df)

prune_df.to_csv(CFG.OUTDIR / "prune_decision_after_hc_nm_slsqp.csv", index=False)

keep_keys: ['016', '021', '028', '032', '036', '039', '040', '042', '044', '046', '048', '050', '051']
drop_keys: ['054', '055']


,key,hc_weight,nm_weight,slsqp_weight,max_weight,min_weight,protected,decision
0,016,0.027616,2.779172e-02,0.027870,2.787003e-02,0.027616,False,keep
1,021,0.013062,1.309052e-02,0.013108,1.310802e-02,0.013062,True,keep
2,028,0.035237,3.530104e-02,0.035428,3.542787e-02,0.035237,True,keep
3,032,0.103038,1.027584e-01,0.102631,1.030385e-01,0.102631,True,keep
4,036,0.027789,2.796491e-02,0.027980,2.798007e-02,0.027789,False,keep
5,039,0.235039,2.351843e-01,0.235346,2.353463e-01,0.235039,False,keep
6,040,0.027732,2.781596e-02,0.027906,2.790626e-02,0.027732,False,keep
7,042,0.076585,7.644337e-02,0.076462,7.658454e-02,0.076443,False,keep
8,044,0.099785,9.971013e-02,0.099637,9.978474e-02,0.099637,False,keep
9,046,0.086579,8.672469e-02,0.086424,8.672469e-02,0.086424,False,keep


In [18]:
# ============================================================
# Rebuild matrices for pruned Ridge / LogReg
# ============================================================

pruned_idx = [model_keys.index(k) for k in keep_keys]

model_keys_full = model_keys.copy()
model_keys_pruned = keep_keys.copy()

X_raw_full = X_raw
T_raw_full = T_raw

X_raw_pruned = X_raw[:, pruned_idx]
T_raw_pruned = T_raw[:, pruned_idx]

# Temporarily switch global model_keys for meta feature names
model_keys = model_keys_pruned
X_meta, T_meta, meta_feature_names = build_meta_features(
    X_raw_pruned,
    T_raw_pruned,
    keep_keys,
)

print("Pruned X_raw:", X_raw_pruned.shape)
print("Pruned X_meta:", X_meta.shape)
print("Pruned model_keys:", model_keys)

Pruned X_raw: (439140, 13)
Pruned X_meta: (439140, 39)
Pruned model_keys: ['016', '021', '028', '032', '036', '039', '040', '042', '044', '046', '048', '050', '051']


In [19]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_pruned",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


# elastic_best, elastic_hist = run_meta_regressor_two_stage(
#     name="elasticnet_meta_all",
#     estimator_factory=lambda p: ElasticNet(
#         alpha=p["alpha"],
#         l1_ratio=p["l1_ratio"],
#         max_iter=30000,
#         random_state=CFG.SEED,
#         selection="cyclic",
#     ),
#     coarse_grid=elastic_coarse_grid,
#     refine_builder=build_elastic_refined_grid_fast,
# )


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_pruned",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_pruned coarse search
{'alpha': np.float64(0.001)} 0.954729906836005
{'alpha': np.float64(0.0021544346900318843)} 0.9547299086904443
{'alpha': np.float64(0.004641588833612777)} 0.9547299115534382
{'alpha': np.float64(0.01)} 0.9547299154900551
{'alpha': np.float64(0.021544346900318832)} 0.9547299262913508
{'alpha': np.float64(0.046415888336127774)} 0.954729951798025
{'alpha': np.float64(0.1)} 0.9547300021606928
{'alpha': np.float64(0.21544346900318823)} 0.9547300996976942
{'alpha': np.float64(0.46415888336127775)} 0.9547303317954153
{'alpha': np.float64(1.0)} 0.9547308209119196
{'alpha': np.float64(2.154434690031882)} 0.9547318000884147
{'alpha': np.float64(4.6415888336127775)} 0.9547336399199584
{'alpha': np.float64(10.0)} 0.9547370225148804
{'alpha': np.float64(21.54434690031882)} 0.9547419110446664
{'alpha': np.float64(46.41588833612773)} 0.9547478287557412
{'alpha': np.float64(100.0)} 0.9547520914610574
{'alpha': np.float64(21

,stage,score,alpha
30,refined,0.954752,117.461894
15,coarse,0.954752,100.000000
29,refined,0.954752,100.000000
31,refined,0.954752,137.972966
28,refined,0.954752,85.133992
32,refined,0.954751,162.065660
27,refined,0.954751,72.477966
26,refined,0.954750,61.703386
33,refined,0.954750,190.365394
25,refined,0.954749,52.530556



logreg_meta_pruned coarse search
{'C': np.float64(0.001)} 0.9549951257869735
{'C': np.float64(0.0021544346900318843)} 0.9549956935382117
{'C': np.float64(0.004641588833612777)} 0.9550011034257189
{'C': np.float64(0.01)} 0.955002863874246
{'C': np.float64(0.021544346900318832)} 0.9550016430024928
{'C': np.float64(0.046415888336127774)} 0.9550002623886862
{'C': np.float64(0.1)} 0.9549996385487922
{'C': np.float64(0.21544346900318823)} 0.954999291671037
{'C': np.float64(0.46415888336127775)} 0.9549984940994591
{'C': np.float64(1.0)} 0.9549983993278498
{'C': np.float64(2.154434690031882)} 0.9550001688208356
{'C': np.float64(4.6415888336127775)} 0.9550001547661375
{'C': np.float64(10.0)} 0.9550001571736552
{'C': np.float64(21.54434690031882)} 0.9550001516103372
{'C': np.float64(46.41588833612773)} 0.9550001504065785
{'C': np.float64(100.0)} 0.9550001493980238
{'C': np.float64(215.44346900318823)} 0.9550001500812383
{'C': np.float64(464.1588833612773)} 0.9550001501137722
{'C': np.float64(10

,stage,score,C
27,refined,0.955003,0.007248
28,refined,0.955003,0.008513
3,coarse,0.955003,0.010000
29,refined,0.955003,0.010000
30,refined,0.955003,0.011746
31,refined,0.955002,0.013797
26,refined,0.955002,0.006170
33,refined,0.955002,0.019037
32,refined,0.955002,0.016207
4,coarse,0.955002,0.021544


In [20]:
# ============================================================
# D additions: LogReg topK and feature-set variants
# ============================================================
#
# Purpose:
# - Absorb only the low-cost ideas from stacking_vibe_coding.
# - Do not introduce full multi-layer stacking.
#
# Added candidates:
# - logreg_meta_pruned_top3_equal / top5_equal from raw+rank+logit C-search
# - logreg variants: logit_only, raw_logit, raw_only
# - ridge variants: logit_only, raw_logit
# ============================================================

def fit_predict_logreg_meta_features(X_feat, T_feat, C):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_feat[va_idx])[:, 1]

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict_proba(T_feat)[:, 1]

    return oof_meta, test_meta, auc(y, oof_meta)


def fit_predict_ridge_meta_features(X_feat, T_feat, alpha):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            Ridge(
                alpha=alpha,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_feat[va_idx])

    final_model = make_pipeline(
        StandardScaler(),
        Ridge(
            alpha=alpha,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict(T_feat)

    return oof_meta, test_meta, auc(y, oof_meta)


def run_logreg_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nLogReg feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"C": c} for c in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"C": c}
        for c in make_refined_grid_1d(
            best["params"]["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"logreg_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: LogisticRegression on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df


def run_ridge_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nRidge feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"alpha": a} for a in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best["params"]["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"ridge_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="ridge",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: Ridge on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df

if RUN_D_EXTRA:
    # logreg topK / feature variants / ridge variants
    print("\n" + "=" * 80)
    print("D additions: LogReg topK and feature-set variants")
    print("=" * 80)


    # ----------------------------
    # LogReg topK equal blend using the baseline raw+rank+logit search history
    # ----------------------------
    
    def add_logreg_topk_equal_from_history(k):
        hist = logreg_hist.sort_values("score", ascending=False).copy()
        top = hist.drop_duplicates(subset=["C"]).head(k)
    
        oofs_top = []
        preds_top = []
        rows = []
    
        for _, row in top.iterrows():
            C = float(row["C"])
            oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_meta, T_meta, C)
            oofs_top.append(oof_meta)
            preds_top.append(test_meta)
            rows.append({"C": C, "cv_auc_recomputed": float(score), "source_score": float(row["score"])})
    
        oof_avg = np.mean(oofs_top, axis=0)
        pred_avg = np.mean(preds_top, axis=0)
    
        name = f"logreg_meta_pruned_top{k}_equal"
        add_candidate(
            name=name,
            method="logreg_topk_equal",
            oof_pred=oof_avg,
            test_pred=pred_avg,
            params={
                "k": k,
                "feature_variant": "raw_rank_logit",
                "Cs": [float(r["C"]) for r in rows],
            },
            notes=f"D top{k} equal blend of LogisticRegression C candidates on raw+rank+logit features",
        )
    
        top_df = pd.DataFrame(rows)
        top_df.to_csv(CFG.OUTDIR / f"{name}_components.csv", index=False)
        display(top_df)
    
        return top_df
    
    
    logreg_top3_components = add_logreg_topk_equal_from_history(3)
    logreg_top5_components = add_logreg_topk_equal_from_history(5)
    
    
    # ----------------------------
    # LogReg feature-set variants
    # ----------------------------
    
    logreg_variant_results = {}
    for variant in ["logit_only", "raw_logit", "raw_only"]:
        best_v, hist_v = run_logreg_two_stage_on_feature_variant(variant)
        logreg_variant_results[variant] = {
            "best": best_v,
            "history": hist_v,
        }
    
    
    # ----------------------------
    # Ridge feature-set variants
    # ----------------------------
    
    ridge_variant_results = {}
    for variant in ["logit_only", "raw_logit"]:
        best_v, hist_v = run_ridge_two_stage_on_feature_variant(variant)
        ridge_variant_results[variant] = {
            "best": best_v,
            "history": hist_v,
        }


In [21]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    weight_keys = rec.get("weight_keys") or model_keys
    for k, w in zip(weight_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_039,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
2,nm_softmax_raw,nm,0.955026,0.000880,0.000880,"[0.02779172, 0.01309052, 0.03530104, 0.1027583...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000115,0.996272,0.000137,0.997331
1,hc_nonnegative_raw,hc,0.955026,0.000880,0.000880,"[0.0276156, 0.01306189, 0.0352365, 0.10303847,...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000115,0.996272,0.000137,0.997331
3,slsqp_signed_raw_neg005,slsqp_signed,0.955026,0.000880,0.000880,"[0.02787003, 0.01310802, 0.03542787, 0.1026311...","{""constraint"": ""sum1_signed"", ""neg_bound"": -0....",signed SLSQP on raw OOF predictions; high over...,0.000115,0.996271,0.000137,0.997330
5,logreg_meta_pruned,logreg,0.955003,0.000857,0.000857,None,"{""C"": 0.007247796636776954}",two-stage meta CV logistic regression on raw+r...,0.000032,0.990529,0.000037,0.992381
0,avg,avg,0.954945,0.000799,0.000799,"[0.06666667, 0.06666667, 0.06666667, 0.0666666...",{},"simple average of ['016', '021', '028', '032',...",0.000106,0.996105,0.000121,0.997135
4,ridge_meta_pruned,ridge,0.954752,0.000606,0.000606,None,"{""alpha"": 117.46189430880185}",two-stage meta CV on raw+rank+logit features,-0.067929,1.002992,-0.052911,0.994027


,candidate,method,cv_auc,w_016,w_021,w_028,w_032,w_036,w_039,w_040,w_042,w_044,w_046,w_048,w_050,w_051,w_054,w_055
2,nm_softmax_raw,nm,0.955026,0.027792,0.013091,0.035301,0.102758,0.027965,0.235184,0.027816,0.076443,0.099710,0.086725,0.038490,0.089747,0.129294,9.045095e-09,0.009684
1,hc_nonnegative_raw,hc,0.955026,0.027616,0.013062,0.035237,0.103038,0.027789,0.235039,0.027732,0.076585,0.099785,0.086579,0.038629,0.089912,0.129224,0.000000e+00,0.009775
3,slsqp_signed_raw_neg005,slsqp_signed,0.955026,0.027870,0.013108,0.035428,0.102631,0.027980,0.235346,0.027906,0.076462,0.099637,0.086424,0.038349,0.090007,0.129311,-3.260771e-05,0.009572
0,avg,avg,0.954945,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,6.666667e-02,0.066667


In [22]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg 0.9549451780580529 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_avg_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
hc_nonnegative_raw 0.9550259644947208 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_hc_nonnegative_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
nm_softmax_raw 0.9550259897736568 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_nm_softmax_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
slsqp_signed_raw_neg005 0.9550259476746308 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_slsqp_signed_raw_neg005_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
ridge_meta_pruned 0.9547523062506784 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_ridge_meta_pruned_exp_20260516_033_blend_logreg_topk_feature_

In [23]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-16",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys_full": model_keys_full if "model_keys_full" in globals() else model_keys,
        "model_keys_pruned": model_keys_pruned if "model_keys_pruned" in globals() else model_keys,
        "model_keys_current": model_keys,
        "n_models_full": len(model_keys_full) if "model_keys_full" in globals() else len(model_keys),
        "n_models_pruned": len(model_keys_pruned) if "model_keys_pruned" in globals() else len(model_keys),
        "D_additions": [
            "logreg_meta_pruned_top3_equal",
            "logreg_meta_pruned_top5_equal",
            "logreg_meta_pruned_logit_only",
            "logreg_meta_pruned_raw_logit",
            "logreg_meta_pruned_raw_only",
            "ridge_meta_pruned_logit_only",
            "ridge_meta_pruned_raw_logit"
        ],
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "D experiment: low-cost blend-code improvements only.",
        "Avg/HC/NM/SLSQP are raw full-stack methods before pruning.",
        "Ridge/LogReg baseline and variants are run on the pruned stack.",
        "Added LogReg top3/top5 equal blends and feature-set variants: logit_only, raw_logit, raw_only.",
        "Added Ridge feature-set variants: logit_only, raw_logit."
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.9550259897736568, 'delta_vs_039': 0.0008800301981609637, 'delta_vs_best_single': 0.0008800301981609637, 'weights': '[0.02779172, 0.01309052, 0.03530104, 0.10275839, 0.02796491, 0.23518428, 0.02781596, 0.07644337, 0.09971013, 0.08672469, 0.03848996, 0.0897474, 0.12929402, 1e-08, 0.0096836]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 276, "fun": -0.9550259897736568}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 0.00011521471977854666, 'oof_max': 0.9962716905884867, 'pred_min': 0.00013664464129445203, 'pred_max': 0.997331014155954}
